<a href="https://colab.research.google.com/github/Yangximin-2007/finance-ai-lab/blob/main/RAG%E5%AE%9E%E8%B7%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

:
需要安装的库

三个核心库：

LangChain 负责链路编排，

Chroma 负责向量存储，

sentence-transformers 负责把文字转向量（本地免费）。

In [11]:
!pip install langchain langchain-community langchain-text-splitters
!pip install chromadb
!pip install sentence-transformers
!pip install pdfplumber

In [12]:
import chromadb
from sentence_transformers import SentenceTransformer
print("环境就绪")

环境就绪


嵌入模型选择

用 BAAI/bge-small-zh-v1.5，专门为中文优化，模型小（约 100MB），本地跑不需要联网，金融中文文本效果好。

In [13]:
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
test = embed_model.encode(["贵州茅台营业收入增长"])
print(f"向量维度: {test.shape}")  # 应该输出 (1, 512)

向量维度: (1, 512)


In [30]:
# 在存入真实PDF之前先跑这个
# 确保在运行此单元格之前，ILPAtzxyCPGg 单元格已运行，并且 client 对象已初始化。
if 'client' in globals() and client is not None:
    client.delete_collection("finance_docs")
    # 重新获取 collection 对象，确保其指向清空后的状态
    collection = client.get_or_create_collection("finance_docs")
    print("已清空，重新开始")
else:
    print("错误：'client' 对象未初始化。请先运行包含 ChromaDB 客户端初始化的单元格 (ILPAtzxyCPGg)。")

已清空，重新开始


In [31]:
text = load_pdf("/content/drive/MyDrive/finance-docs/贵州茅台2025年年度报告.pdf")
docs = split_text(text, ticker="600519.SH")
add_docs_to_chroma(docs)
print(f"真实文档已存入，共 {len(docs)} 个块")

已存入 343 个块
真实文档已存入，共 343 个块


In [32]:
from google.colab import drive
drive.mount('/content/drive')

# 把 PDF 放在你的 Google Drive 里，比如新建一个 finance-docs 文件夹
# 然后这样访问
text = load_pdf("/content/drive/MyDrive/finance-docs/贵州茅台2025年年度报告.pdf")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


为什么要切块

一份研报几万字，不能整篇塞给模型。切成 500 字左右的块，检索时找最相关的几块，精准又省 token。

In [33]:
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_pdf(path: str) -> str:
    """读取 PDF 研报，返回纯文本"""
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

def split_text(text: str, ticker: str) -> list:
    """切块，每块附带元数据"""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,      # 每块500字
        chunk_overlap=50,    # 相邻块重叠50字，防止切断关键信息
        separators=["\n\n", "\n", "。", "，"]
    )
    chunks = splitter.split_text(text)
    # 每个块附带来源信息，检索后知道来自哪篇文档
    docs = [{
        "text": chunk,
        "metadata": {"ticker": ticker, "source": "research_report"}
    } for chunk in chunks]
    return docs

# 如果暂时没有 PDF，用文本模拟
#sample_text = """
#贵州茅台2024年报摘要：
#公司实现营业总收入1741亿元，同比增长15.7%。
#归母净利润857亿元，同比增长15.4%。
#直销渠道占比提升至45%，i茅台平台全年贡献收入约110亿元。
##风险提示：宏观经济下行压力、消费需求收缩、渠道库存积压。
#"""
#docs = split_text(sample_text, "600519.SH")
print(f"切出 {len(docs)} 个块")

切出 1 个块


In [34]:
results = search_docs("茅台2024年的主要风险是什么？", ticker="600519.SH")
print(results)

['质量发展夯实基础、蓄能增势。有序推进茅台酒“十四五”产能扩建、中华6万吨勾调中心、茅\n台酒用原料储备库、坛厂包装物流园项目一期工程、制酒车间（二期）排水管网维修改造等项目\n建设。建立健全项目安全隐患排查治理长效机制，定期组织安全检查和隐患排查。强化投资精准\n性与项目全过程管理，切实提升服务生产经营、支撑市场战略落地保障能力。\n八是加强风险防控体系建设。深入推进安全生产治本攻坚三年行动和标准化行动，常态化开\n展安全生产隐患排查和专项整治，加强安全宣传培训，强化应急保障能力，不断提升本质安全水\n平。加强预算刚性约束，严控一般性支出，推动实物资产管理体系运行及改进，强化资金管理各\n环节风险防范。进一步完善法治体系建设，稳步推进风险防范、合同管理、合规管理、市场维权、\n普法宣传等重点任务。统筹抓好国安、保密、信访等工作，努力营造和谐稳定良好氛围。\n(四)可能面对的风险\n√适用 □不适用\n一是宏观经济风险；二是安全风险；三是舆情风险；四是环境保护风险。\n(五)其他\n□适用 √不适用\n七、公司因不适用准则规定或国家秘密、商业秘密等特殊原因，未按准则披露的情况和原因说明\n□适用 √不适用', '2023年5月6日，公司第三届董事会第四次会议，审议通过了《关于出资参与设立产业发展\n基金的议案》，为了提升资金收益率，为全体股东创造价值，公司决定出资参与设立两只产业发\n展基金，分别是茅台招华（贵州）产业发展基金合伙企业（有限合伙）（以下简称茅台招华基金）\n和茅台金石（贵州）产业发展基金合伙企业（有限合伙）（以下简称茅台金石基金）。\n公司以自有资金参与设立茅台招华基金和茅台金石基金，认缴出资额各为人民币50亿元。根\n据基金设立协议约定，茅台招华基金和茅台金石基金采用认缴资本制，投资期限为五年，公司将\n在投资期内分三期履行实缴出资义务。\n2023年度，公司已按投资进度对茅台招华基金及茅台金石基金分别完成首期实缴出资人民币\n20亿元。\n截至资产负债表日，公司对茅台招华基金和茅台金石基金尚未履行的认缴出资余额均为人民\n币30亿元。\n135/143贵州茅台酒股份有限公司2025年年度报告\n2、 或有事项\n(1).资产负债表日存在的重要或有事项\n□适用 √不适用\n(2).公司没有需要披露的重要或有事项，也应予以说明：\n□适用 √不适

向量库的作用

把文字块转成向量存进 Chroma。查询时也把问题转成向量，找距离最近的几个块——这就是"语义搜索"。

In [35]:
import chromadb
from sentence_transformers import SentenceTransformer
import os

embed_model = SentenceTransformer("BAAI/bge-small-zh-v1.5")

# 将 ChromaDB 存储路径更改为 Colab 本地目录，以避免 Google Drive FUSE 的写入问题
chroma_db_dir = "/content/chroma_db"

# 确保 ChromaDB 目录存在
os.makedirs(chroma_db_dir, exist_ok=True)

print(f"ChromaDB 存储路径: {chroma_db_dir}")
print(f"目录是否存在: {os.path.isdir(chroma_db_dir)}")
print(f"目录是否可写: {os.access(chroma_db_dir, os.W_OK)}")

client = chromadb.PersistentClient(path=chroma_db_dir)  # 本地持久化
collection = client.get_or_create_collection("finance_docs")

def add_docs_to_chroma(docs: list):
    """把切好的块存入向量库"""
    texts = [d["text"] for d in docs]
    metadatas = [d["metadata"] for d in docs]
    embeddings = embed_model.encode(texts).tolist()
    ids = [f"doc_{i}_{hash(t)}" for i, t in enumerate(texts)]

    collection.upsert(  # upsert = 存在则更新，不存在则插入
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    print(f"已存入 {len(docs)} 个块")

def search_docs(query: str, ticker: str = None, top_k: int = 3) -> list:
    """语义搜索，返回最相关的 top_k 个块"""
    query_embedding = embed_model.encode([query]).tolist()
    where = {"ticker": ticker} if ticker else None
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where=where
    )
    return results["documents"][0]

# 存入并测试检索
# add_docs_to_chroma(docs) # This line will be executed by gYEp63SzNUE0
# results = search_docs("茅台的风险有哪些", ticker="600519.SH") # This line will be executed by gYEp63SzNUE0
# print(results) # This line will be executed by gYEp63SzNUE0

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB 存储路径: /content/chroma_db
目录是否存在: True
目录是否可写: True


In [36]:
results = search_docs("茅台2024年的主要风险是什么？", ticker="600519.SH")
print(results)

['质量发展夯实基础、蓄能增势。有序推进茅台酒“十四五”产能扩建、中华6万吨勾调中心、茅\n台酒用原料储备库、坛厂包装物流园项目一期工程、制酒车间（二期）排水管网维修改造等项目\n建设。建立健全项目安全隐患排查治理长效机制，定期组织安全检查和隐患排查。强化投资精准\n性与项目全过程管理，切实提升服务生产经营、支撑市场战略落地保障能力。\n八是加强风险防控体系建设。深入推进安全生产治本攻坚三年行动和标准化行动，常态化开\n展安全生产隐患排查和专项整治，加强安全宣传培训，强化应急保障能力，不断提升本质安全水\n平。加强预算刚性约束，严控一般性支出，推动实物资产管理体系运行及改进，强化资金管理各\n环节风险防范。进一步完善法治体系建设，稳步推进风险防范、合同管理、合规管理、市场维权、\n普法宣传等重点任务。统筹抓好国安、保密、信访等工作，努力营造和谐稳定良好氛围。\n(四)可能面对的风险\n√适用 □不适用\n一是宏观经济风险；二是安全风险；三是舆情风险；四是环境保护风险。\n(五)其他\n□适用 √不适用\n七、公司因不适用准则规定或国家秘密、商业秘密等特殊原因，未按准则披露的情况和原因说明\n□适用 √不适用', '2023年5月6日，公司第三届董事会第四次会议，审议通过了《关于出资参与设立产业发展\n基金的议案》，为了提升资金收益率，为全体股东创造价值，公司决定出资参与设立两只产业发\n展基金，分别是茅台招华（贵州）产业发展基金合伙企业（有限合伙）（以下简称茅台招华基金）\n和茅台金石（贵州）产业发展基金合伙企业（有限合伙）（以下简称茅台金石基金）。\n公司以自有资金参与设立茅台招华基金和茅台金石基金，认缴出资额各为人民币50亿元。根\n据基金设立协议约定，茅台招华基金和茅台金石基金采用认缴资本制，投资期限为五年，公司将\n在投资期内分三期履行实缴出资义务。\n2023年度，公司已按投资进度对茅台招华基金及茅台金石基金分别完成首期实缴出资人民币\n20亿元。\n截至资产负债表日，公司对茅台招华基金和茅台金石基金尚未履行的认缴出资余额均为人民\n币30亿元。\n135/143贵州茅台酒股份有限公司2025年年度报告\n2、 或有事项\n(1).资产负债表日存在的重要或有事项\n□适用 √不适用\n(2).公司没有需要披露的重要或有事项，也应予以说明：\n□适用 √不适

最后一步：

把检索结果塞给大模型
检索到相关段落后，把它们拼进 prompt，让模型基于这些内容回答，而不是凭空生成。

In [44]:
from openai import OpenAI

client_llm = OpenAI(
    api_key="sk-08a428014fdc437c9afd4a7f4709ea7d",
    base_url="https://api.deepseek.com"
)

def rag_query(question: str, ticker: str) -> str:
    """完整 RAG 链路：检索 → 组装 prompt → 生成"""

    # Step 1: 检索相关文档块
    context_chunks = search_docs(question, ticker=ticker, top_k=3)
    context = "\n\n---\n\n".join(context_chunks)

    # Step 2: 组装 prompt
    system_prompt = """你是专业的金融分析师。
根据提供的研报/财报片段回答问题。
要求：
1. 只基于给定文本回答，不要添加文本中没有的信息
2. 如果文本中找不到答案，明确说明"文档中未提及"
3. 引用具体数据时注明来源片段"""

    user_prompt = f"""参考文档：
{context}从最细致客观严谨的角度对贵州茅台2025年的年度报告进行总结归纳

问题：{question}"""

    # Step 3: 调用大模型
    response = client_llm.chat.completions.create(
        model="deepseek-reasoner",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.6,
        max_tokens=1200
    )
    return response.choices[0].message.content

# 测试
answer = rag_query("茅台2025年的主要风险是什么？", "600519.SH")
print(answer)

根据提供的文档片段，贵州茅台2025年年度报告中列出的主要风险包括：一是宏观经济风险；二是安全风险；三是舆情风险；四是环境保护风险。文档中未提及任何关于未来股价趋势的信息，因此无法回答股价走势。


本阶段产出：finance_rag.py


把前四步封装成一个文件，这就是你简历上的第一个金融 AI 项目核心代码。

In [45]:
# 1. 初始化（只需运行一次）
embed_model = SentenceTransformer("BAAI/bge-small-zh-v1.5")
client_chroma = chromadb.PersistentClient(path="./chroma_db")
collection = client_chroma.get_or_create_collection("finance_docs")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [47]:
# 2. 加入新文档（有新研报就调一次）
text = load_pdf("/content/drive/MyDrive/finance-docs/贵州茅台2025年年度报告.pdf")
docs = split_text(text, ticker="600519.SH")
add_docs_to_chroma(docs)

已存入 343 个块


In [48]:
answer = rag_query("茅台直销渠道占比是多少？", "600519.SH")
print(answer)

根据提供的文档内容，茅台直销渠道（自营和“i茅台”等数字营销平台）的本期销售收入为8,454,303.19万元，批发代理渠道的本期销售收入为8,423,155.33万元，总销售收入为16,877,458.52万元。经计算，直销渠道收入占比约为50.09%（8,454,303.19 / 16,877,458.52 × 100%）。文档中未提及任何关于贵州茅台未来股价趋势的信息，因此无法回答该问题。


In [49]:
answer = rag_query("主要风险有哪些？", "600519.SH")
print(answer)

根据提供的贵州茅台2025年年度报告片段，主要风险包括：

- **宏观经济风险**（见“可能面对的风险”部分）
- **安全风险**（同上）
- **舆情风险**（同上）
- **环境保护风险**（同上）

此外，年报中“与金融工具相关的风险”部分还提及了信用风险、流动性风险、汇率风险及利率风险，但这些属于金融工具层面的风险，而非公司列示的“可能面对的风险”类别。

关于贵州茅台未来股价趋势，文档中未提及任何相关信息，因此无法判断。


推入Github

In [51]:
# 运行这个，会弹出授权页面
from google.colab import userdata
import subprocess

# 配置 git 身份（只需设置一次）
subprocess.run(['git', 'config', '--global', 'user.email', 'ximifctn@163.com'])
subprocess.run(['git', 'config', '--global', 'user.name', 'Yangximin-2007'])

CompletedProcess(args=['git', 'config', '--global', 'user.name', 'Yangximin-2007'], returncode=0)

In [56]:
token = userdata.get('GITHUB')  # 存进 Colab Secrets

!git clone https://{token}@github.com/Yangximin-2007/finance-ai-lab.git
%cd finance-ai-lab

Cloning into 'finance-ai-lab'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 103 (delta 42), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 111.00 KiB | 4.44 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/finance-ai-lab


In [57]:
# 把你的代码文件复制进仓库目录
!cp /content/RAG实践.ipynb .

# 推送
!git add .
!git commit -m "add RAG pipeline - lesson 3"
!git push

cp: cannot stat '/content/RAG实践.ipynb': No such file or directory
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
remote: Permission to Yangximin-2007/finance-ai-lab.git denied to Yangximin-2007.
fatal: unable to access 'https://github.com/Yangximin-2007/finance-ai-lab.git/': The requested URL returned error: 403
